# How random are random numbers?

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

This demo drops random points
$(x,y)$ into a square, one per mouse click — where each point takes **two
successive draws** from the random-number generator (RNG), `x = random()` then
`y = random()` — so you can *look* at the cloud and judge whether your RNG is any
good. This notebook keeps that exact idea and turns it into an experiment.

The catch: a modern RNG **always looks fine**, so on its own the demo can never
show a generator *failing*. To actually *illustrate how random random numbers
are*, we compare two generators:

- a **good** one — Python's built-in `random`, a **Mersenne Twister**;
- a deliberately **bad** one — a small **linear congruential generator (LCG)**.

Plotted the very way the original demo does it (consecutive draws as $x,y$), the
good generator fills the square evenly while the bad one collapses onto a handful
of **diagonal lines**. We then explain *why*, and show the sting in the tail:
the bad generator still **passes a simple 1-D uniformity test** — because being
uniform is necessary but *not sufficient* for being random.

Only stdlib `math`/`random` plus **matplotlib**/**numpy** are needed.

## 1. Imports

In [ ]:
# Colab-friendly install guard (only matplotlib/numpy are 3rd-party)
import importlib.util, subprocess, sys
for _pkg in ("matplotlib", "numpy"):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

import math
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib.animation import FuncAnimation

rc('animation', html='jshtml', embed_limit=64)   # MB; keep all animation frames
%matplotlib inline

## 2. Two random-number generators

**The good one — Mersenne Twister.** This is what `random.random()` uses. Its
period is $2^{19937}-1$ and it passes essentially every standard statistical
test. We just wrap it so both generators expose the same method: return a float
in $[0,1)$.

**The bad one — a linear congruential generator (LCG).** An LCG makes the next
integer from the previous one,

$$x_{n+1} = (a\,x_n + c)\bmod m,\qquad u_n = x_n/m,$$

and is completely determined by the constants $a$ (multiplier), $c$ (increment)
and $m$ (modulus). LCGs are fast and were once ubiquitous, but with a poorly
chosen (small) multiplier their consecutive outputs are strongly correlated:
successive values fall on a few parallel lines in 2-D (and planes in 3-D — the
notorious **RANDU** is the textbook case). We pick $a=1229,\ c=1,\ m=2048$, which
is uniform in 1-D but blatantly structured in 2-D.

In [ ]:
class MersenneTwister:
    '''The 'good' generator: Python's built-in random (Mersenne Twister).'''
    name = "Mersenne Twister (Python built-in)"

    def __init__(self, seed=100):
        self._rng = random.Random(seed)

    def random(self):
        return self._rng.random()


class BadLCG:
    '''The 'bad' generator: a small linear congruential generator
    x_{n+1} = (a*x_n + c) mod m,  u_n = x_n / m.

    a=1229, c=1, m=2048 is uniform in 1-D but its consecutive pairs fall on
    only a few diagonal lines. Try RANDU (a=65539, c=0, m=2**31) too.'''
    name = "Bad LCG  (a=1229, c=1, m=2048)"

    def __init__(self, seed=1, a=1229, c=1, m=2048):
        self.a, self.c, self.m = a, c, m
        self.state = seed % m

    def random(self):
        self.state = (self.a * self.state + self.c) % self.m
        return self.state / self.m


def draw_xy(gen, n):
    '''Generate n points the way the original demo does: each point is TWO
    successive draws, x = gen.random(), y = gen.random(). Returns arrays x, y.'''
    x = np.empty(n); y = np.empty(n)
    for i in range(n):
        x[i] = gen.random()
        y[i] = gen.random()
    return x, y

## 3. Parameters

`npoints` is how many points to drop in the box (each consumes two RNG draws, as
in the original). `Seed` keeps the good generator reproducible; the bad LCG's
constants live on the `BadLCG` class above — change them to see how sensitive an
LCG's quality is to its parameters.

In [ ]:
npoints = 4000     # number of (x, y) points to generate
Seed    = 100      # seed for the good generator (reproducibility)
BoxDim  = 500.0    # box size, matching the original Tkinter demo (cosmetic)

good = MersenneTwister(seed=Seed)
bad  = BadLCG(seed=1)

gx, gy = draw_xy(good, npoints)   # good generator
bx, by = draw_xy(bad,  npoints)   # bad generator
print(f"good: {good.name}")
print(f"bad : {bad.name}")

## 4. The visual test — exactly what the original demo does

Drop `npoints` points in the box, each one made of two successive draws
($x=$ `random()`, $y=$ `random()`), and just look. The **good** generator scatters
evenly over the whole square. The **bad** generator — plotted in precisely the
same way — piles its points onto a small set of **diagonal lines**. That lattice
is the fingerprint of a linear congruential generator, and the original demo would
have shown it too, had it been fed this RNG.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
for a, X, Y, gen, col in ((ax[0], gx, gy, good, "#1f77b4"),
                          (ax[1], bx, by, bad,  "#d62728")):
    a.scatter(X * BoxDim, Y * BoxDim, s=6, c=col, alpha=0.5, edgecolors="none")
    a.set_title(gen.name, fontsize=10)
    a.set_xlim(0, BoxDim); a.set_ylim(0, BoxDim)
    a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"Random points in the box  ({npoints} points, 2 draws each)\n"
             "good = evenly filled     bad = collapses onto diagonal lines",
             fontsize=12)
plt.tight_layout(); plt.show()

## 5. Why the lattice? Successive numbers are not independent

Randomness is more than filling the square: successive numbers must also be
**independent**. But an LCG is a *deterministic recurrence* — each value is a fixed
function of the previous one, $u_{n+1} = (a\,u_n\,m + c)\bmod m\,/\,m$. When the
multiplier $a$ is small, that function is nearly a straight line (with wrap-around),
so a point $(u_n, u_{n+1})$ and hence $(x, y)$ can only ever land on one of $a$-ish
parallel lines.

Plotting each value against the one that immediately follows it, $(u_i, u_{i+1})$,
isolates this cleanly (the good generator stays a shapeless cloud):

In [ ]:
# a single stream from each generator to form consecutive pairs
def stream(gen, n):
    return np.array([gen.random() for _ in range(n)])

g = MersenneTwister(seed=Seed); u_good = stream(g, 2 * npoints)
b = BadLCG(seed=1);             u_bad  = stream(b, 2 * npoints)

fig, ax = plt.subplots(1, 2, figsize=(11, 5.4))
for a, u, gen in ((ax[0], u_good, good), (ax[1], u_bad, bad)):
    a.scatter(u[:-1], u[1:], s=6, c="#7f7f7f", alpha=0.5, edgecolors="none")
    a.set_title(gen.name, fontsize=10)
    a.set_xlim(0, 1); a.set_ylim(0, 1); a.set_aspect("equal")
    a.set_xlabel(r"$u_i$"); a.set_ylabel(r"$u_{i+1}$")
fig.suptitle(r"Consecutive pairs $(u_i,\,u_{i+1})$ — the bad LCG lies on a lattice",
             fontsize=12)
plt.tight_layout(); plt.show()

## 6. The sting: uniform ≠ random

Here is the trap. If you only check whether the numbers are **uniform in 1-D** —
bin them and count — the bad LCG looks *perfect*. Uniformity is necessary but not
sufficient; it says nothing about the *order* of the numbers.

Two quantities:

- **Uniformity ($\chi^2$).** Bin into $k$ equal bins; a uniform generator gives
  $\chi^2\approx k-1$. Both generators pass.
- **Lag-1 autocorrelation.** The correlation between $u_i$ and $u_{i+1}$; near $0$
  for independent numbers. The bad LCG's lattice shows up here as a clearly
  non-zero value — the test that *does* catch it.

In [ ]:
def chi2_uniform(u, k=20):
    counts, _ = np.histogram(u, bins=k, range=(0, 1))
    expected = len(u) / k
    return float(np.sum((counts - expected) ** 2 / expected))

def lag1_autocorr(u):
    a, b = u[:-1], u[1:]
    a = a - a.mean(); b = b - b.mean()
    return float(np.sum(a * b) / math.sqrt(np.sum(a * a) * np.sum(b * b)))

print(f"{'generator':38s} {'chi^2 (~k-1=19)':>16s} {'lag-1 autocorr (~0)':>22s}")
for gen, u in ((good, u_good), (bad, u_bad)):
    print(f"{gen.name:38s} {chi2_uniform(u):16.2f} {lag1_autocorr(u):22.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, u, gen in ((ax[0], u_good, good), (ax[1], u_bad, bad)):
    a.hist(u, bins=20, range=(0, 1), color="#2ca02c", edgecolor="white")
    a.axhline(len(u) / 20, color="k", ls="--", lw=1, label="ideal")
    a.set_title(gen.name, fontsize=10); a.set_xlabel("value"); a.legend(fontsize=8)
ax[0].set_ylabel("count")
fig.suptitle("1-D uniformity — BOTH fill the histogram; uniform is not the same as random",
             fontsize=12)
plt.tight_layout(); plt.show()

## 7. Filling the box, point by point

The original demo's live animation: watch each generator drop its points. The good
one sprays over the whole square; the bad one lays every point down on the same few
lines from the very first draws.

In [ ]:
stride = max(1, npoints // 120)     # cap ~120 frames
frames = list(range(stride, npoints + 1, stride))

fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
scats = []
for a, gen, col in ((ax[0], good, "#1f77b4"), (ax[1], bad, "#d62728")):
    s = a.scatter([], [], s=6, c=col, alpha=0.5, edgecolors="none")
    scats.append(s)
    a.set_title(gen.name, fontsize=10)
    a.set_xlim(0, BoxDim); a.set_ylim(0, BoxDim); a.set_aspect("equal")
    a.set_xticks([]); a.set_yticks([])
sup = fig.suptitle("", fontsize=12)
plt.tight_layout()

def update(n):
    scats[0].set_offsets(np.column_stack((gx[:n] * BoxDim, gy[:n] * BoxDim)))
    scats[1].set_offsets(np.column_stack((bx[:n] * BoxDim, by[:n] * BoxDim)))
    sup.set_text(f"{n} points dropped")
    return scats

anim = FuncAnimation(fig, update, frames=frames, interval=80, blit=False)
plt.close(fig)
anim

## 8. Take-home messages

- **The simple visual test works — if you plot consecutive draws.** Because each
  point uses two successive numbers, the original demo already puts $(u_i,u_{i+1})$
  on screen, which is exactly the plot that exposes an LCG's lattice.
- **Independence is the hard part.** A generator can fill the square (and a 1-D
  histogram) yet have every value be a near-deterministic function of the last.
- **Uniform ≠ random.** The bad LCG passes a $\chi^2$ uniformity check but fails the
  lag-1 autocorrelation and the 2-D pair test. Passing *one* test is never proof;
  real RNGs are vetted against whole batteries (Diehard, TestU01, …).
- **Higher dimensions are stricter.** Some LCGs look fine as 2-D pairs but betray
  themselves as 3-D triples $(u_i,u_{i+1},u_{i+2})$ — how RANDU was caught. Set
  `BadLCG`'s constants to RANDU (`a=65539, c=0, m=2**31`) and compare 2-D vs 3-D.
- Modern generators (Mersenne Twister, PCG, xoshiro) are engineered to defeat
  exactly these attacks — which is why the "good" panel is reassuringly boring.